# Refinement trials and the paired test

A refinement is a trial: one change, compared with the current incumbent on the same folds, and promoted only if the gain is larger than
the run-to-run spread. This chapter keeps the register of every trial — and says plainly which of its trials were chosen after their
scores had already been seen.


## Overview

1. The Nadeau–Bengio corrected paired t-test, one-sided, and what "corrected" means.
2. The promotion rule, the Bonferroni line printed beside it, and the trial budget.
3. The register: the three rungs, then two threshold trials — logistic and boosting at the base rate — and the selection effect they carry.
4. A paired test on a pair you choose.

Shared code lives in `lensml.py` beside the chapters (parameters, transforms, ladder, scoring, paired test, register); `show()` prints a
function's source where a chapter introduces it.

The README's glossary defines the terms used here (ladder, rung, incumbent, register, base rate, wet-call share).


## Prerequisites

| Concepts | Importance | Notes |
|---|---|---|
| [Chapter 4](04-baselines-and-the-ladder.ipynb) | Helpful | the ladder and the threshold table |
| A paired t-test | Helpful | a mean difference over its standard error |
| [scikit-learn: statistical comparison of models](https://scikit-learn.org/stable/auto_examples/model_selection/plot_grid_search_stats.html) | Helpful | the source of the test |

- **Time to learn**: about 25 minutes; the trials take about a minute.
- **System requirements**: the `environment.yml` at the repository root; no account, no local data beyond the committed cache.


## Imports


In [ ]:
import pandas as pd
from lensml import (
    FEATURES,
    MAX_TESTED_TRIALS,
    TRIALS,
    Register,
    at_base_rate,
    compare,
    compute_corrected_ttest,
    corrected_std,
    recall_vs_calls,
    setup,
    show,
    show_p,
    verdict,
)
from scipy.stats import t as t_dist

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 70)

In [ ]:
S = setup()  # the cache, the test members, the transforms (chapters 1-2)
train, test = S.train, S.test
X_tr, y_tr = train[FEATURES].values, train["wet"].values
g_tr = train["member_id"].values
print(
    f"train: {S.n_train_members} members / {len(train)} winters   "
    f"test: {test.member_id.nunique()} members / {len(test)} winters   "
    f"N_FOLDS = {S.n_folds}"
)

## The test

Unidata's instruction is to run trials "until … they no longer improve". Improvement is judged against the **run-to-run spread**: each
trial repeats the member-grouped folds `N_REPEATS` times with different fold draws, and both models are scored on identical draws. Scores
from overlapping training sets are not independent, so a plain paired t-test under-estimates the variance. This cookbook uses the
**Nadeau–Bengio corrected paired t-test**, from scikit-learn's example *Statistical comparison of models using grid search*
(BSD-3-Clause, credited in `lensml.py`).

**"Corrected" means the standard error, not multiplicity.** A plain paired t-test divides the variance of the `k` per-fold
differences by `k`; `corrected_std` multiplies it by `1/k + n_test/n_train` instead, to account for the overlap between training sets;
it does nothing about running many tests. The test is **one-sided**: a candidate must
*beat* the incumbent, so a loss prints `p` near 1 (a change from the example, which took the absolute value under a right-tailed label).
The register prints `_sd`, the plain per-fold SD of the differences, for scale; `_t` divides the gap by the corrected error, and `_t` is
the test.


In [ ]:
show(corrected_std, compute_corrected_ttest, compare)

## The rule, the Bonferroni line, the budget

Recall alone rewards calling wet always; precision alone rewards calling it almost never; AP cannot see a threshold. So a candidate is
**promoted only if one of the two numbers the decision uses improves measurably — `p < 0.05` with a positive gap — while the other does not
fall on average** (mean gap ≥ 0). Both tests sit in every register row, with the wet-call share (`called_wet_cand`, `called_wet_inc`)
beside recall.

The example warns that "if multiple models are statistically compared, a multiple comparisons correction is needed". The register does
not apply one — each `p` is per test, uncorrected — so it **bounds the looks** at `MAX_TESTED_TRIALS` rows, two metrics each, and prints the
**Bonferroni line** (0.05 ÷ the looks so far) under every register. A `p` under 0.05 but above that line is a promotion under the
uncorrected rule only. The line counts **only the register's own looks**: not chapter 3's three contrast arms, not chapter 4's table and
threshold rows, not any look you take outside this register.

**What the rule cannot see.** It compares a candidate with the *incumbent* only, and the first incumbent is the sign rule. If the sign
rule's precision sits below the always-wet bar (chapter 4), a candidate with no skill at all — precision merely *at* the base rate — can
beat it. Read such a promotion as a promotion over the sign rule, not as skill; chapter 6 reads the tested model against the base rate of
the same winters.


In [ ]:
show(verdict, Register.record)

## The ladder climb

The three rungs, each against the incumbent. They were fixed in chapter 4 before any score was read.


In [ ]:
reg = Register(S)
for name, make_model, what_changed in TRIALS[:3]:  # rungs 1-3
    reg.record(name, make_model, what_changed)
print(
    f"incumbent after the ladder ({len(reg.rows)} trials): {reg.incumbent_name}"
)
reg.bonferroni_line()
reg.table()

## Two threshold trials, chosen after the fact

At the default 0.5 the probabilistic rungs almost never call wet, so the next two trials lower a threshold to the designed base rate.
**Neither was blind.** Logistic at the base rate is the second row of chapter 4's threshold table, printed before it is registered here.
Boosting at the base rate is exactly chapter 3's contrast model: arm A scored it on the same seeds and folds, and its precision and recall
there are the `precision_cand` and `recall_cand` this register prints for trial 5. It was registered *because* that validation score had
been seen. So its `p` is optimistic in a way no correction in this register can fix: the looks that picked it are not among the looks the
Bonferroni line counts. Chapters 6 and 7 are the remaining checks; chapter 6's test members were scored during development, so it is provisional.


In [ ]:
show(at_base_rate)
reg.record(*TRIALS[3])  # logistic@base-rate
print(f"incumbent after {len(reg.rows)} trials: {reg.incumbent_name}")
reg.bonferroni_line()
reg.table().tail(1)

In [ ]:
reg.record(*TRIALS[4])  # boosting@base-rate
print(f"incumbent after {len(reg.rows)} trials: {reg.incumbent_name}")
reg.bonferroni_line()
recall_vs_calls(reg, "boosting@base-rate")
reg.table().tail(1)

Read the row in two halves. A random extra wet call catches a wet winter at the base rate, so a classifier with no skill has recall equal
to its wet-call share: the `accounts for` line prints how much of the recall gain the change in share alone would give, and how much is
left for the ranking. Precision is the half that speaks to skill. Place each `p` against the Bonferroni line — and remember that the line
does not count the look that chose this trial.

## Your trial

One tested row is left in the budget. **Write it down before you run anything that scores it**: what changes (one thing), why, and which
metric you expect to move. Candidates: a threshold on the tree rung, a tree depth, a leaf size, or the boosting learning rate moved one
notch. Add the trial to `YOUR_TRIALS` below, run the cell, and read the new row against the spread and the Bonferroni line. Then copy the
same `YOUR_TRIALS` into chapter 6, which rebuilds this register before its test.


In [ ]:
YOUR_TRIALS = [
    # ('tree@base-rate', at_base_rate('tree'), 'tree, threshold lowered to the base rate'),
]
for t in YOUR_TRIALS:
    reg.record(*t)  # a name already recorded is skipped
left = MAX_TESTED_TRIALS - len(reg.rows)
print(
    f"incumbent after {len(reg.rows)} trials: {reg.incumbent_name}   "
    f"(tested rows left: {left})"
)
reg.bonferroni_line()
reg.table()

## A paired test on a pair you choose

The cell below re-runs the test on one pair from the register — by default the last promotion, on the metric it was promoted on. The test
is right-tailed, so a pair whose `PAIR_B` lost prints `p` near 1; the cell then reads the left tail and names the loss. Change `PAIR_A`,
`PAIR_B` or `HAND_METRIC` to any names in the register, and state whether the lead is a measured difference, and what you keep if it is
not.


In [ ]:
promoted = [r for r in reg.rows if r["verdict"].startswith("promoted")]
last = promoted[-1] if promoted else reg.rows[-1]
PAIR_A, PAIR_B = (
    last["incumbent"],
    last["candidate"],
)  # any two names in the register
promoted_on = (
    last["verdict"].split(":")[1].split()[0]
    if last["verdict"].startswith("promoted")
    else None
)
HAND_METRIC = promoted_on or "precision"
print(
    f"default pair: register row {last['trial']} ({last['candidate']} vs "
    f"{last['incumbent']}, verdict '{last['verdict']}') → metric {HAND_METRIC}"
    + ("" if promoted_on else "  (nothing promoted: precision by default)")
)

n_test = len(train) // S.n_folds
n_train = len(train) - n_test  # exactly as compare() sets them
a, b = reg.scores[PAIR_A][HAND_METRIC], reg.scores[PAIR_B][HAND_METRIC]
diff = b - a  # candidate minus incumbent
df_hand = len(diff) - 1
t_hand, p_hand = compute_corrected_ttest(
    diff, df=df_hand, n_train=n_train, n_test=n_test
)
p_loss = t_dist.cdf(t_hand, df_hand)  # the LEFT tail: small when PAIR_B lost
print(
    f"{PAIR_B} − {PAIR_A} on {HAND_METRIC}: {len(diff)} paired fold scores   "
    f"n_train = {n_train}   n_test = {n_test}"
)
print(
    f"means {b.mean():.3f} vs {a.mean():.3f}   gap {diff.mean():+.4f}   per-fold SD of "
    f"the gap {diff.std(ddof=1):.4f}   corrected standard error "
    f"{corrected_std(diff, n_train, n_test):.4f}  (the spread the test uses)"
)
if p_hand < 0.05:
    reading = (
        f"p < 0.05: a measured GAIN for {PAIR_B} under the uncorrected rule; "
        "read it against the Bonferroni line below"
    )
elif p_loss < 0.05:
    reading = (
        f"a measured LOSS for {PAIR_B}: {abs(t_hand):.1f} corrected standard errors "
        f"(left-tail p = {show_p(p_loss)})"
    )
else:
    reading = "NOT a measured difference (neither tail under 0.05)"
print(f"t = {t_hand:.2f}   p = {show_p(p_hand)}   →  {reading}")
reg.bonferroni_line()

```{admonition} Try it: read the register
:class: tip
For every row, say whether its gap is outside the corrected spread, and whether any promotion survives the Bonferroni line. Then compare
the corrected standard error printed by the scaffold with the per-fold SD beside it: how much did the correction widen the error a plain
paired t would have used?
```

## Summary

Every refinement is a recorded trial with a one-sided paired test whose standard error is corrected for overlapping folds. Promotion needs a
measured gain on one decision metric with no average loss on the other; the uncorrected `p` is printed beside the Bonferroni line, which
counts only the register's looks. The two threshold trials were chosen after their validation scores were seen, so their promotions are
leads to test, not findings. The register's last promotion is the model chapter 6 tests.

### What's next?

[Chapter 6](06-test-and-justify.ipynb) runs the held-out test once on the register's final incumbent and asks whether it deserves to be
used.

## Resources and references

| Resource | Where it bears | Licence |
|---|---|---|
| scikit-learn example, *Statistical comparison of models using grid search*. https://scikit-learn.org/stable/auto_examples/model_selection/plot_grid_search_stats.html | the corrected paired t-test; the multiple-comparisons warning | BSD-3-Clause |
| NSF Unidata, *Supervised Machine Learning Readiness*. https://github.com/Unidata/supervised-ml-readiness | refine until trials no longer improve | Apache-2.0 |
